<a href="https://colab.research.google.com/github/yasmeenfayyaz/AIML_8_FailureSensorPredictor_byte/blob/main/aiml_8_failuresensorpredictor_byte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 08**


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Automatically find the training file inside the content folder
train_file_path = None
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file == 'train_FD001.txt':
            train_file_path = os.path.join(root, file)
            break
    if train_file_path:
        break

if not train_file_path:
    print("Error: train_file_path.txt not found. Please check if the CMaps folder is uploaded.")
else:
    print(f"File successfully found at: {train_file_path}")

    # 2. Load the dataset (Space-separated values)
    df = pd.read_csv(train_file_path, sep=r'\s+', header=None)

    # Drop extra/empty columns if any
    df = df.dropna(axis=1, how='all')

    # CMaps standard column naming (2 identifiers + 3 settings + remaining sensors)
    base_cols = ['unit_number', 'time_cycles', 'setting_1', 'setting_2', 'setting_3']
    sensor_cols = [f'sensor_{i}' for i in range(1, df.shape[1] - 4)]
    df.columns = base_cols + sensor_cols

    # 3. Create Remaining Useful Life (RUL) and Binary Target (Failure within 30 cycles)
    max_cycles = df.groupby('unit_number')['time_cycles'].max().reset_index()
    max_cycles.columns = ['unit_number', 'max']
    df = df.merge(max_cycles, on='unit_number', how='left')
    df['RUL'] = df['max'] - df['time_cycles']
    df['failure'] = (df['RUL'] <= 30).astype(int)

    # 4. Feature Engineering: Rolling window statistics (Mean & Std over 12 cycles)
    selected_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12']
    features_to_roll = [s for s in selected_sensors if s in df.columns]

    if not features_to_roll:
        features_to_roll = sensor_cols[:3] # Fallback if specific sensors are missing

    for col in features_to_roll:
        df[f'{col}_rolling_mean'] = df.groupby('unit_number')[col].transform(lambda x: x.rolling(window=12, min_periods=1).mean())
        df[f'{col}_rolling_std'] = df.groupby('unit_number')[col].transform(lambda x: x.rolling(window=12, min_periods=1).std()).fillna(0)

    # 5. Select Features (X) and Target (y)
    feature_cols = [c for c in df.columns if 'rolling' in c or c in sensor_cols]
    X = df[feature_cols]
    y = df['failure']

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 6. Train the Model
    print("Model training has started...")
    model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # 7. Evaluate and Print Metrics
    y_pred = model.predict(X_test)
    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred))

    # 8. Save the Model Artifact
    joblib.dump(model, 'predictive_maintenance_model.pkl')
    print("\nModel successfully saved as: predictive_maintenance_model.pkl")

    # 9. Save Confusion Matrix Plot
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Precision & Recall Focus')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig('evaluation_metrics.png')
    plt.close()
    print("Evaluation chart successfully saved as: evaluation_metrics.png")

File successfully found at: /content/train_FD001.txt
Model training has started...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      3543
           1       0.92      0.85      0.89       584

    accuracy                           0.97      4127
   macro avg       0.95      0.92      0.93      4127
weighted avg       0.97      0.97      0.97      4127


Model successfully saved as: predictive_maintenance_model.pkl
Evaluation chart successfully saved as: evaluation_metrics.png
